## Classification: Logistic Regression

The full details on logistic regression require deeper study, but we can gain some insight from looking at the function we are trying to fit to our data. Plot out the curve to the following equation, called the **logistic function**:

$$ f(x) = \frac{e^{x}} {1+e^{x} }  \equiv \frac{1}{1+e^{-x} } $$

&#9989; **Do This**: Create a logistic function and then make the plot of $f(x)$ for $x$ over the range -6 to 6.

In [ ]:
import numpy as np

def fn(x):
    return ??


x = np.arange(-6,6,.1)
y = fn(x)
plt.plot(x,y)
plt.xlabel('x')
plt.ylabel('y');

What is interesting about that curve is that all values of x are mapped into the range for y of 0.0-1.0. Assuming you have a binary classifier, that is one that only has two class labels, that is the mapping you want: all combination of features map into the two class labels 0, or 1. Moreover, the graph looks a lot like a [cumulative probability distribution](https://en.wikipedia.org/wiki/Cumulative_distribution_function). The probability that a set of features is of class 1 is 1 to the right and 0 to the left. As you watched in the pre-class video, this is the basis for logistic regression.

It's considered a regression because the "x" in our logisitic function is actually going to be a regression equation, such that

$$ fn= b_{0} + b_{1}x_{1} + b_{2}x_{2} + \ldots  $$

for as many terms as we like and the new logistic function

$$ f(x) = \frac{e^{fn(x)}} {1+e^{fn(x)} }  \equiv \frac{1}{1+e^{-fn(x)} } $$

Logisitic classification tries to find the values for the parameters $b_{i}$ that gives maximal performance on training, and hopefully testing. Let's let `statsmodels` do that.

We are going to use all the training data from above and train a logistic regression. It is similar to what we did before with regular regression. Here, we will use the bindary iris data again.


In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

iris_df = pd.read_csv("binary-iris.csv")

iris_targets = iris_df.values[:,-1].astype(str) 
iris_data = iris_df.values[:,:-1].astype(float)
label_map = {
    "Iris-setosa": 0,
    "Iris-versicolor": 1}

y_numeric = np.array([label_map[i] for i in iris_targets])
print(y_numeric[:10])

- train-test split

In [ ]:
class_labels= y_numeric
feature_vectors = iris_data

train_vectors, test_vectors, train_labels, test_labels = train_test_split(feature_vectors, class_labels, test_size=0.25)
print(train_vectors)
print(train_labels)

Note, very importantly, the use of `sm.add_constant` on the training vectors. We talked about that when we did OLS in statsmodels. That column of constant is what the $b_{0}$ or intercept will train against. We need that column to get an intercept. (Note that this code requires that you used `train_labels` and `train_vectors` when you did the train-test split earlier)

- Logistic regression model from `statsmodels`
  

In [ ]:
import statsmodels.api as sm
from sklearn import metrics

logit_model = sm.Logit(train_labels, sm.add_constant(train_vectors))
result = logit_model.fit()
# result = logit_model.fit_regularized(alpha=1.0)
print(result.summary() )

The "Pseudo R-squ" is the equivalent (mostly) of the R-squared value in Linear regression that we looked for before. It ranges from 0 (poor fit) to 1 (perfect fit). The P values under "P > |z|" are measures of significance. 

### How'd it go?

There are a number of ways that we can check the performance of our model and we will continue new ways throughout the semester. The major difference in the standard statistics approach and supervised learning approaches is that we test our models using the data that we held out: "the testing data." 

That is, we will use our classifier model to make predictions from the test features and we can then compare those predictions to actual test labels. To test accuracy, we can use the output of the `.fit()` method of the model to predict how well the classifier works on the test data (the data it was not trained on). Conveniently that is the `.predict()` method and, again, we use it on the result of the `.fit()`. 

**Note:** The output from `.predict()` is not a 0/1 value as the test labels are, but rather a fraction between 0 and 1 indicating how likely each entry is to be one class or another. We can make the assumption that anything greater than 0.5 would be a 1 class and anything less than 0.5 would be a 0 class. 

&#9989; **Do This**: do the following:
- use the `.predict()` method (look up the documentation as necessary) to create the predicted labels using the test input
- convert the output of the `.predict()` method to the 0/1 class values of the test labels
- print the resulting predicted class values of the test labels

One of the first metrics we will use in determining how well a machine learning model is working is the "accuracy score", which compares the predictions our model made for the test labels and the actual test labels. This score is one of many metrics we can use and is included in `sklearn.metrics` as (surprise) `accuracy_score()`. Here's the [documentation](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.accuracy_score.html#sklearn.metrics.accuracy_score) on `accuracy_score`.

&#9989; **Do This**: Using the predicted 0/1 labels you just created:
- Use the `sklearn.metrics` we imported at the top and run the `accuracy_score` on the 0/1 predicted label and the test labels.
- Print your accuracy result

In [ ]:
predicted = result.predict(sm.add_constant(test_vectors))
print(predicted)

nominal = [1.0 if x >0.5 else 0.0 for x in predicted]
print(nominal)
print(metrics.accuracy_score(nominal, test_labels))

In [ ]:
from sklearn.metrics import accuracy_score, confusion_matrix, ConfusionMatrixDisplay

cm = confusion_matrix(test_labels, nominal)

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=["Iris-setosa", "Iris-versicolor"])

disp.plot()
plt.title("Logistic Regression on Binary Iris Dataset")
plt.show()

print("Accuracy:", accuracy_score(test_labels, predicted_labels))

- Visualize the decision boundary.

In [ ]:
import matplotlib.pyplot as plt

plt.title("Iris Dataset")
plt.xlabel("Sepal Length")
plt.ylabel("Sepal Width")

for species in ["Iris-setosa","Iris-versicolor"]:
    these_points = iris_data[ iris_targets == species ]
    plt.scatter(these_points[:,0], these_points[:,1], label = species)
    
plt.legend()

m = ??
b = ??


plt.plot(np.linspace(4.5,7,50), [m*x+b for x in np.linspace(4.5,7,50)], "k-")
plt.show()